In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
MODEL_NAME = "cointegrated/rubert-tiny2"
MAX_LEN =64
BATCH_SIZE = 32
EPOCHS = 2
LR = 2e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 333

torch.manual_seed(SEED)

In [ ]:
# =======================
# DATASET
# =======================
class FiveDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.texts = dataframe['text'].tolist()
        self.targets = dataframe['rate'].tolist() if 'rate' in dataframe else None
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = " ".join(str(self.texts[idx]).split())

        inputs = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True
        )

        item = {
            "ids": torch.tensor(inputs["input_ids"], dtype=torch.long),
            "mask": torch.tensor(inputs["attention_mask"], dtype=torch.long),
        }

        if self.targets is not None:
            item["targets"] = torch.tensor(self.targets[idx], dtype=torch.long)

        return item


In [ ]:
# =======================
# MODEL (FIXED)
# =======================
class ModelForClassification(torch.nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size  # 312

        self.dropout = torch.nn.Dropout(dropout)
        self.classifier = torch.nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        last_hidden = outputs.last_hidden_state

        # ===== MEAN POOLING =====
        mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        summed = torch.sum(last_hidden * mask, dim=1)
        summed_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
        mean_pool = summed / summed_mask

        x = self.dropout(mean_pool)
        logits = self.classifier(x)

        return logits

In [ ]:
# =======================
# TRAINER
# =======================
class Trainer:
    def __init__(self, model, train_loader, val_loader, class_weights):
        self.model = model.to(DEVICE)
        self.train_loader = train_loader
        self.val_loader = val_loader

        self.loss_fn = CrossEntropyLoss(
            weight=torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
        )

        self.optimizer = AdamW(
            self.model.parameters(),
            lr=LR,
            weight_decay=1e-5
        )

        total_steps = len(train_loader) * EPOCHS

        self.scheduler = get_linear_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=int(0.1 * total_steps),
            num_training_steps=total_steps
        )

    def train_epoch(self):
        self.model.train()
        losses = []

        for batch in self.train_loader:
            ids = batch["ids"].to(DEVICE)
            mask = batch["mask"].to(DEVICE)
            targets = batch["targets"].to(DEVICE)

            outputs = self.model(ids, mask)
            loss = self.loss_fn(outputs, targets)

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            self.scheduler.step()

            losses.append(loss.item())

        return np.mean(losses)

    def val_epoch(self):
        self.model.eval()

        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in self.val_loader:
                ids = batch["ids"].to(DEVICE)
                mask = batch["mask"].to(DEVICE)
                targets = batch["targets"].to(DEVICE)

                outputs = self.model(ids, mask)

                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(targets.cpu().numpy())

        f1 = f1_score(all_labels, all_preds, average='weighted')

        return f1

    def fit(self):
        print("-" * 30)
        for epoch in range(EPOCHS):
            print(f"Epoch {epoch}")
            train_loss = self.train_epoch()
            val_f1 = self.val_epoch()

            print(f"Epoch {epoch+1}")
            print(f"Train Loss: {train_loss:.4f}")
            print(f"Val F1: {val_f1:.4f}")
            print("-" * 30)


In [ ]:
path = ""  #  "/content/drive/MyDrive/tmp/"

train_data = pd.read_csv(os.path.join(path, "train.csv"))

In [ ]:
train_data.head()

In [ ]:
le = LabelEncoder()
train_data["rate"] = le.fit_transform(train_data["rate"])

# ===== STRATIFIED SPLIT =====
train_df, val_df = train_test_split(
    train_data,
    test_size=0.2,
    stratify=train_data["rate"],
    random_state=SEED
)

# ===== CLASS WEIGHTS =====
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["rate"]),
    y=train_df["rate"]
)

# ===== TOKENIZER =====
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ===== DATASETS =====
train_dataset = FiveDataset(train_df, tokenizer, MAX_LEN)
val_dataset = FiveDataset(val_df, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# ===== MODEL =====
model = ModelForClassification(
    MODEL_NAME,
    num_classes=len(le.classes_)
)

# ===== TRAIN =====
trainer = Trainer(model, train_loader, val_loader, class_weights)



In [ ]:
trainer.fit()

In [ ]:
trainer.save("baseline_model.ckpt")

In [ ]:
trainer = Trainer.load("baseline_model.ckpt")

In [ ]:
test_data = pd.read_csv(os.path.join(path, "test.csv"))

test_dataset = FiveDataset(test_data, tokenizer, MAX_LEN)

test_params = {"batch_size": BATCH_SIZE,
               "shuffle": False,
               "num_workers": 0
               }

test_dataloader = DataLoader(test_dataset, **test_params)

In [ ]:
predictions = t.predict(test_dataloader)

In [ ]:
sample_submission = pd.read_csv(os.path.join(path_gv, "sample_submission.csv"))
sample_submission["rate"] = predictions
sample_submission.rate = le.inverse_transform(sample_submission.rate)
sample_submission.head()

In [ ]:
sample_submission.to_csv(path_gv + "submission_007.csv", index=False)